# 🎭 Viggle-Animate (MiniMax-H3) — Piloto de Testes ComfyUI
**Fase 1: Ambiente de Validação e Benchmark no Google Colab (L4 / A100)**

Este notebook implementa o teste piloto do **Viggle-Animate** (nós ComfyUI de [Saganaki22/ComfyUI-Viggle-Animate-H3](https://github.com/Saganaki22/ComfyUI-Viggle-Animate-H3)), modelo de substituição de personagens e transferência de movimento baseado no fine-tune do MiniMax-H3 (*ref2va*, ~33.1B parâmetros).

> **Aviso de Hardware:** Devido ao peso do modelo int8 (~21 GB), é obrigatório utilizar uma GPU com **pelo menos 22 GB de VRAM** (ex.: NVIDIA L4 de 24 GB ou A100 de 40/80 GB via Google Colab Pro).


## 1. Diagnóstico de Hardware (GPU e VRAM)
Verifica a presença da GPU e a quantidade de VRAM disponível. Se for menor que 22 GB (ex.: T4 gratuita de 15 GB), a execução é interrompida com aviso para usar L4 ou A100.


In [ ]:
# ==============================================================================
# 1. DIAGNÓSTICO DE HARDWARE: GPU E VRAM
# ==============================================================================
import sys
import torch

print("🔍 Verificando GPU disponível no ambiente...")

if not torch.cuda.is_available():
    print("\n" + "="*70)
    print("❌ ERRO: Nenhuma GPU detectada!")
    print("No menu do Colab, vá em: Ambiente de execução -> Alterar tipo de ambiente de execução")
    print("e selecione uma GPU (L4 ou A100 no Colab Pro).")
    print("="*70 + "\n")
    raise RuntimeError("Nenhuma GPU ativa no Google Colab.")

gpu_name = torch.cuda.get_device_name(0)
total_vram_bytes = torch.cuda.get_device_properties(0).total_memory
total_vram_gb = total_vram_bytes / (1024 ** 3)

print("\n" + "="*70)
print(f"🎮 GPU Detectada : {gpu_name}")
print(f"💾 VRAM Total    : {total_vram_gb:.2f} GB")
print("="*70 + "\n")

# Exibe tabela do nvidia-smi
!nvidia-smi

# Verificação de VRAM mínima de segurança (22 GB)
if total_vram_gb < 22.0:
    print("\n" + "!"*70)
    print("🛑 BLOQUEIO DE CAPACIDADE DE MEMÓRIA VRAM:")
    print("O menor modelo tem 21 GB; use L4 ou A100 (Colab Pro)")
    print("!"*70 + "\n")
    raise RuntimeError("O menor modelo tem 21 GB; use L4 ou A100 (Colab Pro)")

print("\n✅ VRAM suficiente detectada! Você pode prosseguir para a próxima célula.")


## 2. Instalar ComfyUI e Custom Nodes
Clona o ComfyUI oficial, instala dependências e configura os repositórios necessários em `custom_nodes`:
- `Saganaki22/ComfyUI-Viggle-Animate-H3` (nós do Viggle Animate)
- `Kosinkadink/ComfyUI-VideoHelperSuite` (carregamento e salvamento de vídeo)
- `kijai/ComfyUI-KJNodes` e `jamesWalker55/comfyui-various` (suporte aos nós auxiliares do workflow de exemplo)


In [ ]:
# ==============================================================================
# 2. INSTALAÇÃO DO COMFYUI E CUSTOM NODES
# ==============================================================================
import os
import subprocess

print("📦 1/3 Instalando ComfyUI oficial...")
%cd /content
if not os.path.exists("/content/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q huggingface_hub

print("\n🧩 2/3 Clonando e configurando Custom Nodes...")
custom_nodes_dir = "/content/ComfyUI/custom_nodes"
os.makedirs(custom_nodes_dir, exist_ok=True)
%cd {custom_nodes_dir}

# 1. ComfyUI-Viggle-Animate-H3 (Saganaki22)
if not os.path.exists("ComfyUI-Viggle-Animate-H3"):
    !git clone https://github.com/Saganaki22/ComfyUI-Viggle-Animate-H3.git
if os.path.exists("ComfyUI-Viggle-Animate-H3/requirements.txt"):
    !pip install -q -r ComfyUI-Viggle-Animate-H3/requirements.txt

# 2. ComfyUI-VideoHelperSuite (VHS - Kosinkadink)
if not os.path.exists("ComfyUI-VideoHelperSuite"):
    !git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
if os.path.exists("ComfyUI-VideoHelperSuite/requirements.txt"):
    !pip install -q -r ComfyUI-VideoHelperSuite/requirements.txt

# 3. ComfyUI-KJNodes (necessário para nós do workflow v1.2.0: CustomSigmas, ModelPreviewOverrideKJ)
if not os.path.exists("ComfyUI-KJNodes"):
    !git clone https://github.com/kijai/ComfyUI-KJNodes.git
if os.path.exists("ComfyUI-KJNodes/requirements.txt"):
    !pip install -q -r ComfyUI-KJNodes/requirements.txt

# 4. comfyui-various (para o nó JWImageResizeToMegapixels presente no workflow)
if not os.path.exists("comfyui-various"):
    !git clone https://github.com/jamesWalker55/comfyui-various.git

%cd /content/ComfyUI
print("\n✅ ComfyUI e nós customizados instalados com sucesso!")


## 3. Baixar Modelos com huggingface_hub
Faz o download dos pesos oficiais apontados no README do `ComfyUI-Viggle-Animate-H3`:
- `drbaph/Viggle-Animate-ComfyUI`: `diffusion_models/minimax_h3_ref2va_viggle_pruned_int8_convrot.safetensors` (~21 GB), `loras/viggle_animate_dmd_lora_r64.safetensors` (~0.94 GB), `text_cond/fixed_embed_fwd_anyframe.safetensors` (~1.5 MB).
- `Kijai/MiniMax-H3-experimental`: `minimax_h3_video_vae_int8_convrot.safetensors` (~3.17 GB).

*Mostra progresso nativo e pula automaticamente arquivos que já estejam no disco.*


In [ ]:
# ==============================================================================
# 3. DOWNLOAD DOS MODELOS VIGGLE-ANIMATE (HUGGING FACE)
# ==============================================================================
import os
import shutil
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()

# Configuração dos modelos conforme a seção 'Model Links' do README:
DOWNLOAD_LIST = [
    {
        "name": "Diffusion Model (int8 pruned, ~21 GB)",
        "repo_id": "drbaph/Viggle-Animate-ComfyUI",
        "repo_file": "diffusion_models/minimax_h3_ref2va_viggle_pruned_int8_convrot.safetensors",
        "target_dir": "/content/ComfyUI/models/diffusion_models",
        "target_file": "minimax_h3_ref2va_viggle_pruned_int8_convrot.safetensors"
    },
    {
        "name": "LoRA DMD 4-Step (r64, ~0.94 GB)",
        "repo_id": "drbaph/Viggle-Animate-ComfyUI",
        "repo_file": "loras/viggle_animate_dmd_lora_r64.safetensors",
        "target_dir": "/content/ComfyUI/models/loras",
        "target_file": "viggle_animate_dmd_lora_r64.safetensors"
    },
    {
        "name": "Text Conditioning Embeddings (~1.5 MB)",
        "repo_id": "drbaph/Viggle-Animate-ComfyUI",
        "repo_file": "text_cond/fixed_embed_fwd_anyframe.safetensors",
        "target_dir": "/content/ComfyUI/models/text_cond",
        "target_file": "fixed_embed_fwd_anyframe.safetensors"
    },
    {
        "name": "Video VAE int8 convrot (~3.17 GB)",
        "repo_id": "Kijai/MiniMax-H3-experimental",
        "repo_file": "minimax_h3_video_vae_int8_convrot.safetensors",
        "target_dir": "/content/ComfyUI/models/vae",
        "target_file": "minimax_h3_video_vae_int8_convrot.safetensors"
    }
]

print("🚀 Verificando e baixando modelos necessários...\n")

for item in DOWNLOAD_LIST:
    repo_id = item["repo_id"]
    repo_file = item["repo_file"]
    target_dir = item["target_dir"]
    target_file = item["target_file"]
    dest_path = os.path.join(target_dir, target_file)

    os.makedirs(target_dir, exist_ok=True)

    # 1. Pular caso já exista e esteja completo
    if os.path.exists(dest_path) and os.path.getsize(dest_path) > 1024:
        size_mb = os.path.getsize(dest_path) / (1024 * 1024)
        print(f"⏩ [PULADO] Já existe: {dest_path} ({size_mb:.2f} MB)")
        continue

    # 2. Conferir com o repositório remoto para evitar adivinhações
    print(f"📡 Checando arquivo '{repo_file}' em '{repo_id}'...")
    try:
        repo_files = api.list_repo_files(repo_id)
    except Exception as e:
        print(f"❌ Erro ao listar repositório {repo_id}: {e}")
        raise

    if repo_file not in repo_files:
        print("\n" + "!"*75)
        print("❌ ARQUIVO NÃO ENCONTRADO NO REPOSITÓRIO!")
        print(f"Repositório: {repo_id}")
        print(f"Arquivo procurado: {repo_file}")
        print("Arquivos disponíveis no repositório:")
        for rf in repo_files:
            print(f"   - {rf}")
        print("!"*75 + "\n")
        raise FileNotFoundError(f"Arquivo '{repo_file}' inexistente em '{repo_id}'.")

    # 3. Baixar exibindo barra de progresso nativa do huggingface_hub
    print(f"⬇️ Baixando {item['name']}...")
    downloaded_path = hf_hub_download(
        repo_id=repo_id,
        filename=repo_file,
        local_dir="/content/hf_cache",
        local_dir_use_symlinks=False
    )

    # Move para o destino correto dentro de models/
    if os.path.abspath(downloaded_path) != os.path.abspath(dest_path):
        shutil.move(downloaded_path, dest_path)

    print(f"✅ Concluído: {dest_path} ({os.path.getsize(dest_path)/(1024*1024):.2f} MB)\n")

print("🎉 Todos os modelos foram verificados e estão prontos nas pastas do ComfyUI!")


## 4. Iniciar ComfyUI e Expor via Túnel Cloudflared
Inicia o processo do ComfyUI com `--listen` na porta 8188 e abre um túnel público `trycloudflare.com` para acesso direto à interface gráfica.


In [ ]:
# ==============================================================================
# 4. INICIALIZAÇÃO DO COMFYUI E TÚNEL PÚBLICO (CLOUDFLARE)
# ==============================================================================
import os
import re
import time
import subprocess
import urllib.request

print("⚙️ Preparando o túnel Cloudflared...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Encerra processos residuais de execuções anteriores
!pkill -f cloudflared || true
!pkill -f "python main.py" || true
time.sleep(1)

# Inicia ComfyUI em segundo plano
print("🚀 Iniciando ComfyUI (--listen 127.0.0.1 --port 8188)...")
comfy_log = open('/content/comfyui.log', 'w')
comfy_proc = subprocess.Popen(
    ['python', '/content/ComfyUI/main.py', '--listen', '127.0.0.1', '--port', '8188'],
    stdout=comfy_log,
    stderr=subprocess.STDOUT
)

# Aguarda inicialização local do ComfyUI
print("⏳ Aguardando ComfyUI responder na porta 8188...")
comfy_ready = False
for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=1) as resp:
            if resp.status == 200:
                comfy_ready = True
                break
    except Exception:
        pass
    time.sleep(1)

if comfy_ready:
    print("✅ ComfyUI ativo localmente!")
else:
    print("⚠️ ComfyUI ainda carregando nós, iniciando túnel em paralelo...")

# Inicia o túnel Cloudflare apontando para o ComfyUI
print("🌐 Abrindo túnel público Cloudflare...")
tunnel_log = open('/content/tunnel.log', 'w')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT
)

tunnel_url = None
for _ in range(40):
    time.sleep(1)
    if os.path.exists('/content/tunnel.log'):
        with open('/content/tunnel.log', 'r') as f:
            content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if match:
                tunnel_url = match.group(0)
                break

if tunnel_url:
    print("\n" + "="*75)
    print("🎉 COMFYUI ONLINE E PRONTO PARA USO!")
    print(f"👉 Link de Acesso: {tunnel_url}")
    print("="*75 + "\n")
    print("Mantenha esta célula em execução para manter o túnel ativo.")
else:
    print("❌ Falha ao obter a URL do túnel. Verifique /content/tunnel.log")


## 5. Copiar Workflow de Exemplo (Single Shot v1.2.0)
Copia o arquivo `viggle-animate-h3_workflow-v1.2.0.json` para o diretório de workflows do ComfyUI e para `/content`, facilitando o carregamento na interface.


In [ ]:
# ==============================================================================
# 5. COPIAR WORKFLOW DE EXEMPLO (viggle-animate-h3_workflow-v1.2.0.json)
# ==============================================================================
import os
import shutil

src_workflow = "/content/ComfyUI/custom_nodes/ComfyUI-Viggle-Animate-H3/example_workflows/viggle-animate-h3_workflow-v1.2.0.json"

dest_dirs = [
    "/content/ComfyUI/user/default/workflows",
    "/content/ComfyUI/workflows",
    "/content"
]

copied_paths = []
if os.path.exists(src_workflow):
    for d in dest_dirs:
        os.makedirs(d, exist_ok=True)
        dst = os.path.join(d, "viggle-animate-h3_workflow-v1.2.0.json")
        shutil.copy2(src_workflow, dst)
        copied_paths.append(dst)
    
    print("✅ Workflow copiado com sucesso!")
    print(f"📂 Origem: {src_workflow}")
    print("📋 Arquivos prontos em:")
    for cp in copied_paths:
        print(f"   - {cp}")
    print("\n💡 Como carregar no ComfyUI:")
    print("1. Na interface do ComfyUI, clique em 'Load' e navegue até o workflow.")
    print("2. Ou simplesmente arraste 'viggle-animate-h3_workflow-v1.2.0.json' do painel do Colab para a tela do ComfyUI.")
else:
    print(f"⚠️ Arquivo de workflow não encontrado em: {src_workflow}")


## 6. Célula de Medição: Tempo de Prompt e Pico de Memória VRAM
Depois de enfileirar o prompt no ComfyUI, execute esta célula. Ela amostra o consumo de VRAM via `nvidia-smi` em tempo real durante a renderização e, ao término, extrai o tempo exato registrado no log do ComfyUI.


In [ ]:
# ==============================================================================
# 6. MEDIÇÃO DE TEMPO DE EXECUÇÃO E PICO DE VRAM (GPU)
# ==============================================================================
import os
import re
import time
import subprocess

def get_current_vram_mb():
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
            text=True
        ).strip()
        return float(out.split('\n')[0])
    except Exception:
        return 0.0

def get_total_vram_mb():
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
            text=True
        ).strip()
        return float(out.split('\n')[0])
    except Exception:
        return 0.0

def find_prompt_times(log_text):
    # Detecta linhas padrão do ComfyUI: 'Prompt executed in XX.XX seconds'
    matches = re.findall(r'Prompt executed in ([\d\.]+) seconds', log_text)
    return [float(m) for m in matches]

total_vram = get_total_vram_mb()
initial_vram = get_current_vram_mb()
peak_vram = initial_vram

print("="*75)
print("📊 MONITOR DE BENCHMARK: VIGGLE-ANIMATE NO COMFYUI")
print(f"VRAM Total da GPU : {total_vram:.0f} MB ({total_vram/1024:.2f} GB)")
print(f"VRAM Inicial      : {initial_vram:.0f} MB ({initial_vram/1024:.2f} GB)")
print("="*75)

log_path = "/content/comfyui.log"
start_pos = 0
if os.path.exists(log_path):
    start_pos = os.path.getsize(log_path)

print("\n🚀 Monitorando em tempo real... Execute ou acompanhe o workflow no ComfyUI.")
print("   (Para encerrar a medição a qualquer momento, clique no botão Stop nesta célula)\n")

last_prompt_time = None
try:
    for _ in range(900):  # Monitora por até 15 minutos
        time.sleep(1.0)
        curr_vram = get_current_vram_mb()
        if curr_vram > peak_vram:
            peak_vram = curr_vram
        
        # Lê novas linhas do log
        if os.path.exists(log_path):
            with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
                f.seek(start_pos)
                new_logs = f.read()
                times = find_prompt_times(new_logs)
                if times:
                    last_prompt_time = times[-1]
                    break
        
        print(f"\r⏳ Monitorando... VRAM Atual: {curr_vram:.0f} MB | Pico: {peak_vram:.0f} MB ({peak_vram/1024:.2f} GB)", end="", flush=True)

except KeyboardInterrupt:
    print("\n⏹️ Medição pausada pelo usuário.")

# Se não pegou no intervalo novo, busca no arquivo completo
if last_prompt_time is None and os.path.exists(log_path):
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        all_logs = f.read()
        all_times = find_prompt_times(all_logs)
        if all_times:
            last_prompt_time = all_times[-1]

print("\n\n" + "="*75)
print("🏁 RELATÓRIO DE DESEMPENHO DO PILOTO")
print("="*75)
if last_prompt_time is not None:
    print(f"⏱️ Tempo de Execução do Prompt : {last_prompt_time:.2f} segundos ({last_prompt_time/60:.2f} minutos)")
else:
    print("⏱️ Tempo de Execução do Prompt : Não detectado no log (certifique-se de que o workflow concluiu)")

print(f"🚀 Pico de Memória VRAM        : {peak_vram:.0f} MB ({peak_vram/1024:.2f} GB / {total_vram/1024:.2f} GB)")

print("\n📋 Últimas 15 linhas do log do ComfyUI:")
if os.path.exists(log_path):
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()
        print("".join(lines[-15:]))
else:
    print("(Arquivo comfyui.log não encontrado)")
print("="*75)
